# Assignment 2 — Sentiment Analysis on Claude 4 Tweets
**Course:** Data Analytics and Visualisation (CSC601)  
**College:** Rizvi College of Engineering, Mumbai  
**Semester:** VI | TE AI&DS (2025-26)  

---
### Topic: Claude 4 (Anthropic)
Claude 4 is Anthropic's latest family of large language models (Haiku / Sonnet / Opus), built on their Constitutional AI framework.  
We classify 100 tweets about Claude 4 as **Positive**, **Neutral**, or **Negative** using three ML classifiers.


## Step 1 — Import Libraries

In [ ]:
import os, re, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    classification_report, ConfusionMatrixDisplay, confusion_matrix
)

print("All libraries imported successfully.")


## Step 2 — Load & Explore Dataset

In [ ]:
df = pd.read_csv(os.path.join("..", "data", "tweets_dataset.csv"))
print(f"Total tweets: {len(df)}")
display(df.head(10))


In [ ]:
# Sentiment distribution
print("Sentiment Distribution:")
print(df["sentiment"].value_counts())

# Bar chart
colors = ["#e74c3c", "#95a5a6", "#2ecc71"]
ax = df["sentiment"].value_counts().plot(
    kind="bar", color=colors, edgecolor="black", figsize=(7, 4)
)
ax.set_title("Tweet Sentiment Distribution", fontsize=14, fontweight="bold")
ax.set_xlabel("Sentiment", fontsize=12)
ax.set_ylabel("Count", fontsize=12)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
for p in ax.patches:
    ax.annotate(str(int(p.get_height())),
                (p.get_x() + p.get_width() / 2, p.get_height() + 0.3),
                ha="center", fontsize=11)
plt.tight_layout()
plt.show()


## Step 3 — Preprocessing

Each tweet is cleaned through the following pipeline:
1. Lowercase all text
2. Remove URLs
3. Remove @mentions
4. Remove #hashtags
5. Remove special characters, numbers, and emojis
6. Normalize whitespace


In [ ]:
def preprocess_tweet(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#\w+", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_tweet"] = df["tweet"].apply(preprocess_tweet)

# Compare before / after
print("Before:", df["tweet"].iloc[0])
print("After :", df["clean_tweet"].iloc[0])


## Step 4 — Label Encoding

In [ ]:
label_map    = {"positive": 2, "neutral": 1, "negative": 0}
label_decode = {v: k for k, v in label_map.items()}
df["label"]  = df["sentiment"].map(label_map)
print(df[["tweet", "sentiment", "label"]].head(5).to_string())


## Step 5 — Train / Test Split (80 / 20)
`stratify=y` preserves class proportions in both sets.


In [ ]:
X = df["clean_tweet"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training set : {len(X_train)} tweets")
print(f"Testing set  : {len(X_test)} tweets")
print(f"\nTest class distribution:")
print(pd.Series(y_test).map(label_decode).value_counts().to_string())


## Step 6 — TF-IDF Vectorization

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=500,
    ngram_range=(1, 2),
    stop_words="english",
    sublinear_tf=True
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf  = vectorizer.transform(X_test)

print(f"Training matrix shape : {X_train_tfidf.shape}")
print(f"Testing matrix shape  : {X_test_tfidf.shape}")
print(f"Vocabulary size       : {len(vectorizer.vocabulary_)} features")


## Step 7 — Train Classifiers & Evaluate

In [ ]:
classifiers = {
    "Naive Bayes"        : MultinomialNB(alpha=0.5),
    "SVM (LinearSVC)"    : LinearSVC(C=0.5, max_iter=2000, random_state=42),
    "Logistic Regression": LogisticRegression(C=0.5, max_iter=1000,
                                               solver="lbfgs", random_state=42),
}

target_names = ["Negative", "Neutral", "Positive"]
results      = {}

for name, clf in classifiers.items():
    clf.fit(X_train_tfidf, y_train)
    y_pred    = clf.predict(X_test_tfidf)
    precision = precision_score(y_test, y_pred, average="weighted", zero_division=0)
    recall    = recall_score(y_test, y_pred, average="weighted", zero_division=0)
    f1        = f1_score(y_test, y_pred, average="weighted", zero_division=0)
    results[name] = {"precision": precision, "recall": recall, "f1": f1, "y_pred": y_pred}

    print(f"\n{'='*55}")
    print(f"  {name}")
    print(f"{'='*55}")
    print(f"  Weighted Precision : {precision:.4f}")
    print(f"  Weighted Recall    : {recall:.4f}")
    print(f"  Weighted F1-Score  : {f1:.4f}")
    print()
    print(classification_report(y_test, y_pred, target_names=target_names, zero_division=0))


## Step 8 — Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("Confusion Matrices — All Classifiers", fontsize=14, fontweight="bold")

for ax, (name, m) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, m["y_pred"])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(name, fontsize=11, fontweight="bold")

plt.tight_layout()
plt.show()


## Step 9 — Classifier Comparison Chart

In [ ]:
names      = list(results.keys())
precisions = [results[n]["precision"] for n in names]
recalls    = [results[n]["recall"]    for n in names]
f1s        = [results[n]["f1"]        for n in names]

x   = np.arange(len(names))
w   = 0.25
fig, ax = plt.subplots(figsize=(9, 5))

b1 = ax.bar(x - w,   precisions, w, label="Precision", color="#3498db", edgecolor="black")
b2 = ax.bar(x,       recalls,    w, label="Recall",    color="#2ecc71", edgecolor="black")
b3 = ax.bar(x + w,   f1s,        w, label="F1-Score",  color="#e74c3c", edgecolor="black")

for bars in [b1, b2, b3]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.005,
                f"{bar.get_height():.2f}",
                ha="center", va="bottom", fontsize=8.5)

ax.set_xticks(x)
ax.set_xticklabels(names, fontsize=11)
ax.set_ylim(0, 0.75)
ax.set_ylabel("Score", fontsize=12)
ax.set_title("Classifier Performance Comparison", fontsize=13, fontweight="bold")
ax.legend(fontsize=11)
ax.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()


## Step 10 — Summary & Conclusion

In [ ]:
print(f"{'Classifier':<25} {'Precision':>10} {'Recall':>10} {'F1-Score':>10}")
print("-" * 57)
for name, m in results.items():
    print(f"{name:<25} {m['precision']:>10.4f} {m['recall']:>10.4f} {m['f1']:>10.4f}")

best = max(results, key=lambda x: results[x]["f1"])
print(f"\nBest Classifier : {best}")
print(f"Best F1-Score   : {results[best]['f1']:.4f}")


### Conclusion

| Classifier | Precision | Recall | F1-Score |
|---|---|---|---|
| **Naïve Bayes** | 0.4917 | 0.5500 | **0.5000** ✅ |
| SVM (LinearSVC) | 0.5286 | 0.5500 | 0.4909 |
| Logistic Regression | 0.1684 | 0.4000 | 0.2370 |

**Naïve Bayes performs best** with a weighted F1-score of 0.50 and 55% accuracy on the 20-tweet test set.

**Why Naïve Bayes wins on small data:**
- With only 80 training samples, its probabilistic priors prevent overfitting
- Excels on sparse, high-dimensional TF-IDF feature spaces
- Achieves perfect recall (1.00) on the Neutral class

**Possible Improvements:**
- Use a larger dataset (1000+ tweets) for better generalization
- Replace TF-IDF with BERT embeddings for semantic understanding
- Apply hyperparameter tuning via GridSearchCV
- Use ensemble/voting classifiers for robustness
